In [ ]:
# The same installs as the MOSS runs that hung, so the loader, safetensors and
# huggingface_hub versions match the session being diagnosed.
!pip -q install "transformers>=5.0,<5.1" accelerate speechbrain faster-whisper jiwer pyloudnorm noisereduce num2words silero-vad onnxruntime


In [ ]:
import os
import re
import statistics
import subprocess
import sys
import time

import torch
import transformers
import transformers.core_model_loading as loading

print("transformers  :", transformers.__version__)
print("torch         :", torch.__version__)
print("cpu count     :", os.cpu_count())
print("loader workers:", loading.GLOBAL_WORKERS)
assert transformers.__version__.startswith("5.0."), "the hang was seen on 5.0.x - test that version"

MODEL_ID = "OpenMOSS-Team/MOSS-TTS-Local-Transformer"
CODEC_ID = "OpenMOSS-Team/MOSS-Audio-Tokenizer"
RESULTS = []
TRIAL_TIMEOUT = 300
WARMUP_TIMEOUT = 1500
SEQUENTIAL_TRIALS = 4
THREADED_TRIALS = 3
TRIAL_SCRIPT = 'import os\nimport sys\nimport time\n\nimport torch\nimport transformers.core_model_loading as loading\n\ndeactivated = loading.is_env_variable_true("HF_DEACTIVATE_ASYNC_LOAD")\nprint(f"FLAG {deactivated}", flush=True)\nif deactivated != (os.environ.get("EXPECT_SEQUENTIAL") == "1"):\n    print("FLAG MISMATCH - transformers did not read the variable as intended", flush=True)\n    sys.exit(3)\n\nfrom transformers import AutoModel, AutoProcessor\n\nMODEL_ID = "OpenMOSS-Team/MOSS-TTS-Local-Transformer"\nstart = time.time()\nAutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)\nmiddle = time.time()\nprint(f"PROCESSOR {middle - start:.1f}", flush=True)\nAutoModel.from_pretrained(\n    MODEL_ID, trust_remote_code=True, torch_dtype=torch.bfloat16, attn_implementation="sdpa"\n)\nprint(f"MODEL {time.time() - middle:.1f}", flush=True)\n'


def _text(stream):
    if stream is None:
        return ""
    return stream if isinstance(stream, str) else stream.decode(errors="replace")


def trial(sequential, index):
    env = dict(os.environ)
    env.pop("HF_DEACTIVATE_ASYNC_LOAD", None)
    env["EXPECT_SEQUENTIAL"] = "1" if sequential else "0"
    env["HF_HUB_OFFLINE"] = "1"
    if sequential:
        env["HF_DEACTIVATE_ASYNC_LOAD"] = "1"
    name = f"{'sequential' if sequential else 'threaded'} #{index}"
    start = time.time()
    try:
        done = subprocess.run([sys.executable, "-c", TRIAL_SCRIPT], env=env, text=True,
                              capture_output=True, timeout=TRIAL_TIMEOUT)
        out, err = done.stdout, done.stderr
        status = "ok" if done.returncode == 0 else f"exit {done.returncode}"
    except subprocess.TimeoutExpired as exc:
        out, err = _text(exc.stdout), _text(exc.stderr)
        status = "HUNG"
    elapsed = time.time() - start
    processor = re.search(r"PROCESSOR ([\d.]+)", out)
    model = re.search(r"MODEL ([\d.]+)", out)
    flag = re.search(r"FLAG (\w+)", out)
    print(f"{name:14s} {status:8s} {elapsed:6.1f}s   "
          f"processor {processor.group(1) + 's' if processor else '-':>7s}   "
          f"model {model.group(1) + 's' if model else '-':>7s}   "
          f"flag read as {flag.group(1) if flag else '?'}", flush=True)
    if status != "ok":
        # The last progress bar says where it stopped; tqdm redraws with \r.
        for line in err.strip().splitlines()[-3:]:
            print("        ", line.split("\r")[-1][-150:], flush=True)
    RESULTS.append({"sequential": sequential, "status": status, "elapsed": elapsed})


In [ ]:
# Download once, outside the timed loads, so each trial measures loading and
# nothing else. Bounded, because a stalled download must not pass for a hang.
download = (
    "import time\n"
    "from huggingface_hub import snapshot_download\n"
    f"for repo in ({MODEL_ID!r}, {CODEC_ID!r}):\n"
    "    t = time.time()\n"
    "    snapshot_download(repo)\n"
    "    print(repo, f'{time.time() - t:.0f}s', flush=True)\n"
)
try:
    subprocess.run([sys.executable, "-c", download], text=True, timeout=WARMUP_TIMEOUT, check=True)
except subprocess.TimeoutExpired:
    raise SystemExit(f"the download did not finish within {WARMUP_TIMEOUT}s - a network stall, "
                     "not the load race, so nothing was tested")


In [ ]:
# The pool disabled, first - before any threaded load has had the chance to
# hang and leave something behind in this session.
for index in range(1, SEQUENTIAL_TRIALS + 1):
    trial(sequential=True, index=index)


In [ ]:
# transformers' default, with the thread pool on: the control for the cell above.
for index in range(1, THREADED_TRIALS + 1):
    trial(sequential=False, index=index)


In [ ]:
def tally(sequential):
    rows = [r for r in RESULTS if r["sequential"] == sequential]
    loaded = [r["elapsed"] for r in rows if r["status"] == "ok"]
    hung = sum(r["status"] == "HUNG" for r in rows)
    return rows, loaded, hung, len(rows) - len(loaded) - hung


seq_rows, seq_ok, seq_hung, seq_err = tally(True)
thr_rows, thr_ok, thr_hung, thr_err = tally(False)
for name, rows, loaded, hung, err in (("sequential", seq_rows, seq_ok, seq_hung, seq_err),
                                      ("threaded", thr_rows, thr_ok, thr_hung, thr_err)):
    median = f", median {statistics.median(loaded):.0f}s" if loaded else ""
    print(f"{name:10s}: {len(loaded)}/{len(rows)} loaded, {hung} hung, {err} errored{median}")

# The rule was fixed before the run, in the notebook's generator.
if seq_hung:
    verdict = "a sequential load hung - the thread pool is not the cause, or not the only one"
elif seq_err:
    verdict = ("sequential loads errored rather than hung - the test itself is broken; "
               "read the error lines above before concluding anything")
elif thr_hung:
    verdict = ("every sequential load succeeded and threaded loading hung in the same session - "
               "HF_DEACTIVATE_ASYNC_LOAD is the fix")
elif thr_err:
    verdict = "threaded loads errored rather than hung - the race was not reproduced; read the errors above"
else:
    verdict = ("inconclusive - nothing hung, so this session did not reproduce the race "
               "and the sequential successes show nothing")
print()
print("VERDICT:", verdict)
